In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
pip install nltk

In [4]:
pip install transformers torch --quiet

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.util import ngrams
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from transformers import BertTokenizer, BertModel
import torch
import plotly.express as px

In [6]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
full_data=pd.read_csv('drive/MyDrive/ML_course/fake_news_full_data.csv', index_col=0)

In [8]:
data=full_data.sample(frac=0.2, random_state=43)

#EDA

In [9]:
data

,title,text,date,is_fake
28815,Top Republican Ryan distances himself from Tru...,"WASHINGTON (Reuters) - Paul Ryan, the top Repu...","October 10, 2016",0
12162,BREAKING! INVESTIGATION: Hillary Clinton Did N...,Was Hillary Clinton negligent or was she doin...,"May 25, 2016",1
28309,Senate Democrats ask Trump attorney general pi...,WASHINGTON (Reuters) - Nine Democratic senator...,"January 17, 2017",0
28080,AWESOME LETTER TO OBAMA: Who is unfit to be pr...,Did anyone else think it was the ultimate iron...,"Aug 5, 2016",1
20758,FRIGHTENING POWER OF THE PRESS: You Won’t Beli...,Remember when the press used to actually repor...,"Jun 18, 2016",1
...,...,...,...,...
10778,The KKK Leader The Media Said ‘Endorsed’ Hill...,Fox News and other right-wing outlets are quiv...,"March 15, 2016",1
21990,Schumer: Comey should still testify to congres...,WASHINGTON (Reuters) - U.S. Senate Democratic ...,"May 18, 2017",0
30984,WATCH: Joy Behar And ‘The View’ PUMMEL Trump ...,The recent CIA assessment concluding that Russ...,"December 12, 2016",1
23757,POP STAR ARIANA GRANDE SAYS: “I hate Americans...,It s interesting how many of these celebrities...,"Jul 8, 2015",1


In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8936 entries, 28815 to 14694
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    8936 non-null   object
 1   text     8936 non-null   object
 2   date     8936 non-null   object
 3   is_fake  8936 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 349.1+ KB


In [11]:
full_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44680 entries, 0 to 44679
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44680 non-null  object
 1   text     44680 non-null  object
 2   date     44680 non-null  object
 3   is_fake  44680 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.7+ MB


In [12]:
data.drop(15914, axis=0, inplace=True)

In [13]:
data['date']=pd.to_datetime(data['date'], format='mixed', dayfirst=True)

In [14]:
data.is_fake.value_counts(normalize=True)

,proportion
is_fake,
1,0.51953
0,0.48047


In [15]:
data['date'].min(), data['date'].max()

(Timestamp('2015-04-05 00:00:00'), Timestamp('2018-02-18 00:00:00'))

In [16]:
data[data['is_fake']==1]['date'].min(), data[data['is_fake']==1]['date'].max()

(Timestamp('2015-04-05 00:00:00'), Timestamp('2018-02-18 00:00:00'))

In [17]:
data[data['is_fake']==0]['date'].min(), data[data['is_fake']==0]['date'].max()

(Timestamp('2016-01-13 00:00:00'), Timestamp('2017-12-31 00:00:00'))

In [18]:
data['len_title']=data['title'].str.len()

In [19]:
data['len_text']=data['text'].str.len()

In [20]:
data.groupby('is_fake')[['len_title','len_text']].describe().T.stack().reset_index()

,level_0,level_1,is_fake,0
0,len_title,count,0,4293.000000
1,len_title,count,1,4642.000000
2,len_title,mean,0,64.578383
3,len_title,mean,1,94.886256
4,len_title,std,0,9.193438
5,len_title,std,1,27.894621
6,len_title,min,0,33.000000
7,len_title,min,1,19.000000
8,len_title,25%,0,59.000000
9,len_title,25%,1,77.000000


In [21]:
a=data.groupby('is_fake')[['len_title','len_text']].describe().T.stack().reset_index()

In [22]:
px.histogram(data_frame=a[a['level_0']=='len_title'], x='level_1', y=0, color='is_fake', barmode='group', barnorm='percent')

In [23]:
px.histogram(data_frame=a[a['level_0']=='len_text'], x='level_1', y=0, color='is_fake', barmode='group', barnorm='percent')

In [24]:
data[(data['len_text']==1)]['is_fake'].value_counts()

,count
is_fake,
1,113


In [25]:
lens_text=data[['len_text','is_fake']].value_counts().reset_index().sort_values(by=['len_text','is_fake'])

In [26]:
px.line(lens_text[(lens_text['len_text']>43)&(lens_text['len_text']<10000)], x='len_text', y='count', facet_row='is_fake')

In [27]:
lens_title=data[['len_title','is_fake']].value_counts().reset_index().sort_values(by=['len_title','is_fake'])

In [28]:
px.line(lens_title, x='len_title', y='count', facet_row='is_fake')

#Text Preprocessing

In [61]:
english_stopwords=stopwords.words('english')
extra_tokens=["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']
english_stopwords.extend(extra_tokens)

In [62]:
# english_stopwords=[i for i in english_stopwords if i not in ["aren't",'couldn',
#  "couldn't",'didn',
#  "didn't", 'doesn',
#  "doesn't",'don',
#  "don't",'hadn',
#  "hadn't",'hasn',
#  "hasn't",'haven',
#  "haven't", 'isn',
#  "isn't",'mightn',
#  "mightn't",'needn',
#  "needn't",'no',
#  'not','shan',
#  "shan't",'shouldn',
#  "shouldn't",'wasn',
#  "wasn't",'weren',
#  "weren't","won't",
#  'wouldn',
#  "wouldn't",]]

In [63]:
def tokenize(text):
  return [word for word in word_tokenize(text.lower()) if word not in english_stopwords]

In [64]:
def get_tag(text):
  tokens=tokenize(text)
  tags=pos_tag(tokens)
  return tags

In [65]:
def get_wordnet_pos(treebank_tag):

    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [66]:
lem=WordNetLemmatizer()
def lemmatization(text):
  tags=get_tag(text)
  lemmatized_words=[]
  for word, tag in tags:
    wordnet_tag=get_wordnet_pos(tag)
    if wordnet_tag:
      lemma=lem.lemmatize(word, wordnet_tag)
    else:
      lemma=word
    lemmatized_words.append(lemma)
  return lemmatized_words


In [67]:
train_idx, test_idx=train_test_split(list(data.index), test_size=0.2, shuffle=True)

In [68]:
data['day']=data['date'].dt.day
data['month']=data['date'].dt.month
data['year']=data['date'].dt.year
data['dayofweek']=data['date'].dt.dayofweek

KeyError: 'date'

In [ ]:
data.drop('date', axis=1, inplace=True)

In [ ]:
train_inputs=data.loc[train_idx].drop('is_fake', axis=1)
train_target=data.loc[train_idx]['is_fake']

In [ ]:
test_inputs=data.loc[test_idx].drop('is_fake', axis=1)
test_target=data.loc[test_idx]['is_fake']

In [ ]:
tfidf_text=TfidfVectorizer(lowercase=True,
                           tokenizer=lemmatization,
                           stop_words=english_stopwords,
                           max_features=1000)

In [ ]:
tfidf_title=TfidfVectorizer(lowercase=True,
                           tokenizer=lemmatization,
                           stop_words=english_stopwords,
                           max_features=1000)

In [ ]:
text_train=pd.DataFrame(tfidf_text.fit_transform(train_inputs['text']).toarray(), columns=tfidf_text.get_feature_names_out(), index=train_inputs.index)
text_test=pd.DataFrame(tfidf_text.transform(test_inputs['text']).toarray(), columns=tfidf_text.get_feature_names_out(), index=test_inputs.index)

In [ ]:
title_train=pd.DataFrame(tfidf_title.fit_transform(train_inputs['title']).toarray(), columns=tfidf_title.get_feature_names_out(), index=train_inputs.index)
title_test=pd.DataFrame(tfidf_title.transform(test_inputs['title']).toarray(), columns=tfidf_title.get_feature_names_out(), index=test_inputs.index)

In [ ]:
title_train

In [ ]:
# train_set=pd.concat([train_inputs, text_train, title_train], axis=1).drop(['title', 'text'], axis=1)
# test_set=pd.concat([test_inputs, text_test, title_test], axis=1).drop(['title', 'text'], axis=1)

In [ ]:
train_set=pd.concat([ text_train, title_train], axis=1)
test_set=pd.concat([text_test, title_test], axis=1)

# Logistic Regression

In [47]:
logreg=LogisticRegression(solver='saga', penalty='l1', max_iter=1000)

In [48]:
def predictions(model, train_input, test_input, train_target, test_target):
  model.fit(train_input, train_target)
  pred=model.predict(test_input)
  accuracy=accuracy_score(test_target, pred)
  f1=f1_score(test_target, pred, average='weighted')
  confusion_m=confusion_matrix(test_target, pred)
  print(f'accurracy score for test dataset : {accuracy}\nf1_score for test dataset : {f1}\nconfusion matrix\n{confusion_m}' )

In [49]:
predictions(logreg, train_set, test_set, train_target, test_target)

accurracy score for test dataset : 0.9988808058198098
f1_score for test dataset : 0.9988808782581968
confusion matrix
[[841   0]
 [  2 944]]


In [52]:
pd.DataFrame(logreg.coef_[0], index=train_set.columns, columns=['importance']).sort_values(by='importance', ascending=False)[:30]

,importance
’,11.705739
:,8.673781
?,6.359204
video,6.286111
trump,4.407480
”,4.075373
obama,3.421319
go,2.535510
clinton,1.487243
@,1.199707


#Random Forest

In [53]:
forest=RandomForestClassifier(n_estimators=100, max_depth=300, random_state=45)

In [54]:
predictions(forest, train_set, test_set, train_target, test_target)

accurracy score for test dataset : 1.0
f1_score for test dataset : 1.0
confusion matrix
[[841   0]
 [  0 946]]


In [55]:
data_sample=full_data.loc[[i for i in full_data.index if i not in train_idx and i not in test_idx]][:2000]

In [56]:
data_sample

,title,text,date,is_fake
0,Earthquake hits off Papua New Guinea,LONDON (Reuters) - A magnitude 5.9 earthquake ...,"September 17, 2017",0
2,Prosecutors say ex-House Speaker Hastert sexua...,(Reuters) - Former U.S. House Speaker Dennis H...,"April 9, 2016",0
3,Romanian protesters halt building of Xmas fair...,BUCHAREST (Reuters) - Romanian protesters clas...,"December 2, 2017",0
4,"Congo elected to U.N. rights council; Britain,...",UNITED NATIONS (Reuters) - Democratic Republic...,"October 16, 2017",0
5,‘Alt-Right’ White House Reporter Gets Fooled ...,When Donald Trump allowed popular Nazi blog Th...,"November 2, 2017",1
...,...,...,...,...
2488,DAN RATHER GOES FULL-ON RADICAL: Media Must Sh...,Dan Rather just released a Facebook post that ...,"Aug 10, 2016",1
2489,Lindsey Graham: Trump Should Be Treated With ...,We re wondering if Sen. Lindsey Graham (R) is ...,"May 13, 2017",1
2490,MEGYN KELLY CONTINUES HER OBSESSION With Trump...,"We can t be certain, but it almost looked like...","Aug 4, 2016",1
2491,TEXAS CIVIL RIGHTS LAWYER and Dartmouth Grad S...,Texas Civil Rights lawyer Rob Ranco is a partn...,"Sep 9, 2017",1


In [57]:
text=pd.DataFrame(tfidf_text.transform(data_sample['text']).toarray(), columns=tfidf_text.get_feature_names_out(), index=data_sample.index)
title=pd.DataFrame(tfidf_title.transform(data_sample['title']).toarray(), columns=tfidf_title.get_feature_names_out(), index=data_sample.index)

In [58]:
sample_test=pd.concat([text, title], axis=1)

In [59]:
sample_target=data_sample['is_fake']

In [60]:
predictions(forest, train_set, sample_test, train_target, sample_target)

accurracy score for test dataset : 0.998
f1_score for test dataset : 0.998
confusion matrix
[[ 953    2]
 [   2 1043]]


#BERT

In [69]:
bert_tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
bert_model=BertModel.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [70]:
input_bert=bert_tokenizer(test_inputs['text'].tolist(), return_tensors='pt', truncation=True, padding=True)

In [ ]:
with torch.no_grad():
  outputs=bert_model(**input_bert)

In [ ]:
input_bert